# ML-KEM-1024 同板性能与可靠性验证

点击 **Kernel → Restart & Run All** 即可完成下列已配置测试，无需复制终端命令。结果随阶段立即保存到本次独立 `evidence/` 子目录，失败记录同样保存。

配对发布包使用同一个 bitstream、100 MHz 时钟和 128 KiB RAM，分别装入 CPU-only 与 Keccak 加速固件。每次换固件先停 CPU、清空全部 RAM、写入镜像、完整读回并核对 SHA-256，然后才允许启动。

默认：两组各三例预检 → 两组各 145 条官方记录 → KeyGen/Encaps/Decaps 各 10 次重复 → 复位、非法输入 magic 和短超时恢复。耗时以板上实际运行为准，阶段开始和结束均显示进度。


## 1. 顶部开关与加载

保留旧加速发布包兼容模式；只有含 `paired_firmware` 的新发布包才执行实板对照。Notebook 模板可以正常保存，不作为不可变运行文件校验。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from html import escape
from IPython.display import HTML, FileLink, display
from mlkem1024 import MLKEM1024

RUN_FULL_145 = True
RUN_REPEATS = True
REPEATS = 10
RUN_RESET_RECOVERY = True
RUN_BAD_MAGIC_RECOVERY = True
RUN_TIMEOUT_RECOVERY = True
CASE_TIMEOUT_SECONDS = 20.0
REPRESENTATIVE_CASES = [0, 1, 115]

BUNDLE = Path.cwd()
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
EVIDENCE = BUNDLE / 'evidence' / STAMP
EVIDENCE.mkdir(parents=True, exist_ok=False)
reports = {}
try:
    board = MLKEM1024.load(BUNDLE / 'release_manifest.json')
except Exception as exc:
    MLKEM1024.export({'stage': 'load', 'passed': False,
                     'error_type': type(exc).__name__, 'error': str(exc)},
                    EVIDENCE / 'load_failure.json')
    print('加载失败，原始错误已保存：', EVIDENCE / 'load_failure.json')
    raise
PAIRED = board.paired_firmware is not None
print('硬件加载成功；模式：', 'CPU-only / 加速 配对验证' if PAIRED else '旧版加速验证')
print('官方记录：', len(board.vectors['cases']), '；时钟：100 MHz；RAM：128 KiB')
print('结果目录：', EVIDENCE)

def stage_result(label, report):
    reports[label] = report
    print(label, '→', report['status'], '|', report.get('report_path', EVIDENCE))
    if not report['passed']:
        print('阶段失败，后续依赖测试停止。请保留结果目录。', report.get('error', '详见原始 JSON'))
    return report['passed']

def old_run(indices, label):
    report = board.run_cases(indices, timeout=CASE_TIMEOUT_SECONDS)
    report['report_path'] = str(board.export(report, EVIDENCE / (label + '.json')))
    return report

def show_table(columns, rows):
    head = '<tr>' + ''.join('<th>' + escape(str(x)) + '</th>' for x in columns) + '</tr>'
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table>' + head + body + '</table>'))


## 2. 三例预检

索引 0、1、115 覆盖 KeyGen、Encaps、Decaps。逐字节核对官方输出、返回码、周期与硬件计数。CPU-only 组全部硬件计数必须为零；加速组计数必须符合对应向量的调用预期。


In [ ]:
print('开始：三例预检……', flush=True)
subset = (board.run_paired(REPRESENTATIVE_CASES, EVIDENCE, timeout=CASE_TIMEOUT_SECONDS, label='subset')
          if PAIRED else old_run(REPRESENTATIVE_CASES, 'subset'))
proceed = stage_result('三例预检', subset)


## 3. 完整 145 条配对测试

每组必须覆盖 145 个不同官方记录，全部通过才显示 `PAIRED_FULL_PASS`。记录包含密钥生成、封装、解封装及官方负向检查，不能解释为 145 次完整协议会话。


In [ ]:
full = None
if RUN_FULL_145 and proceed:
    print('开始：完整 145 条测试；配对模式会依次运行两组……', flush=True)
    full = (board.run_paired(range(145), EVIDENCE, timeout=CASE_TIMEOUT_SECONDS, label='full')
            if PAIRED else old_run(range(145), 'full'))
    proceed = stage_result('完整145条', full)
else:
    print('完整测试未启用或预检失败，已跳过。')


## 4. 重复性与恢复专项

重复统计使用相同三例，各组各 10 次，给出中位数、P95 和最大值。P95 使用最近秩定义；10 个样本时 P95 等于最大值，属于初步稳定性统计。

恢复专项只操作既定控制寄存器和输入邮箱：完成后复位、故意写入错误 magic、用极短主机超时终止未完成任务，随后重新运行正常官方用例。错误 magic 必须产生固件错误码 2，短超时必须确实发生在完成之前。每项都保留注入和恢复记录。这些专项独立于官方向量自带的负向密钥检查。


In [ ]:
repeated = None
recovery = None
if PAIRED and RUN_REPEATS and proceed:
    print('开始：两组代表用例各 %d 次重复……' % REPEATS, flush=True)
    repeated = board.run_repetitions(EVIDENCE, indices=REPRESENTATIVE_CASES,
                                    repeats=REPEATS, timeout=CASE_TIMEOUT_SECONDS)
    proceed = stage_result('重复性', repeated)
checks = []
if RUN_RESET_RECOVERY:
    checks.append('reset')
if RUN_BAD_MAGIC_RECOVERY:
    checks.append('bad_magic')
if RUN_TIMEOUT_RECOVERY:
    checks.append('short_timeout')
if PAIRED and checks and proceed:
    print('开始：复位 / 非法 magic / 短超时恢复专项……', flush=True)
    recovery = board.run_recovery_checks(EVIDENCE, timeout=CASE_TIMEOUT_SECONDS, checks=checks)
    proceed = stage_result('恢复专项', recovery)
if not PAIRED:
    print('当前为旧发布包，重复配对及恢复专项未执行。')
board.reset()
print('本轮执行结束；CPU 已保持复位。当前固件：', board.firmware_mode)


## 5. 同板加速比、数据搬运与波动

**API 加速比** = CPU-only 原始 rdcycle 总和 / 加速原始 rdcycle 总和；**PS 端到端比值** = 两组相同用例的主机事务墙钟总和之比，包含复位、上传、等待、下载和逐字节核验。固件装载、整 RAM 读回与 JSON 写盘单独记录，均不计入上述比值。操作 4 包含种子 KeyGen 再 Decaps。

同一 bitstream 保留 Keccak 硬件，CPU-only 固件不调用它，因此这个实验隔离“启用加速服务”的效果，不用于比较两套设计的面积或功耗。历史 RTL 周期保留在原始记录中，下面的配对比值只使用本轮实板数据。


In [ ]:
measured = full if full is not None else subset
if PAIRED and measured['passed']:
    comparison = measured['comparison']
    print('本轮结果：', measured['status'])
    print('API 原始周期加速比：%.4f x' % comparison['api_speedup'])
    print('API 扣除空计时括号比值：%.4f x' % comparison['api_minus_empty_speedup'])
    print('PS 端到端事务比值：%.4f x' % comparison['ps_wall_speedup'])
    rows = []
    for mode in ('cpu_only', 'accelerated'):
        run = measured['runs'][mode]
        total = comparison['totals'][mode]
        rows.append([mode, run['unique_checked_cases'], total['raw_cycles'],
                     '%.6f' % (total['raw_cycles'] / 1e8),
                     '%.6f' % total['case_wall_seconds'],
                     '%.6f' % sum(c['upload_seconds'] for c in run['cases']),
                     '%.6f' % sum(c['download_seconds'] for c in run['cases']),
                     '%.6f' % total['firmware_load_seconds']])
    show_table(['固件', '通过记录', 'API周期', 'API秒', 'PS事务秒', '上传秒', '下载秒', '固件加载秒(独立)'], rows)
    rows = []
    for op in sorted(set(c['op'] for c in measured['runs']['cpu_only']['cases'])):
        a = [c for c in measured['runs']['cpu_only']['cases'] if c['op'] == op]
        b = [c for c in measured['runs']['accelerated']['cases'] if c['op'] == op]
        rows.append([a[0]['operation'], len(a),
                     '%.4f' % (sum(c['raw_cycles'] for c in a) / sum(c['raw_cycles'] for c in b)),
                     '%.4f' % (sum(c['wall_seconds'] for c in a) / sum(c['wall_seconds'] for c in b))])
    show_table(['操作', '记录数', 'API加速比', 'PS事务比值'], rows)
elif not PAIRED and measured['passed']:
    print(measured['status'], measured['unique_checked_cases'], '/145；当前没有本轮CPU-only对照，不计算实板加速比。')
else:
    print('测试未通过，不生成加速比；请查看失败 JSON。')
if repeated is not None and repeated['passed']:
    rows = []
    for item in repeated['statistics']:
        cycles, wall = item['raw_cycles'], item['wall_seconds']
        rows.append([item['mode'], item['case_index'], item['samples'],
                     cycles['median'], cycles['p95'], cycles['maximum'],
                     '%.3f' % (wall['median'] * 1000), '%.3f' % (wall['p95'] * 1000),
                     '%.3f' % (wall['maximum'] * 1000)])
    show_table(['固件', '索引', '次数', '周期中位数', '周期P95', '周期最大', '墙钟中位数ms', 'P95 ms', '最大ms'], rows)


## 6. 保存与下载

所有阶段原始 JSON 已在执行时保存；这里将本轮证据打包，方便下载后归档。失败时同样可以运行这一单元下载证据。只读下载链接不会再次加载硬件或重跑测试。


In [ ]:
import zipfile
archive = EVIDENCE.parent / (STAMP + '_board_evidence.zip')
with zipfile.ZipFile(str(archive), 'x', compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(EVIDENCE.glob('*.json')):
        output.write(str(path), arcname=path.name)
print('本轮阶段状态：', [(label, report['status']) for label, report in reports.items()])
print('下载本轮全部原始证据：')
display(FileLink(str(archive.relative_to(BUNDLE))))
for label, report in reports.items():
    print(label)
    display(FileLink(str(Path(report['report_path']).relative_to(BUNDLE))))


实板通过表示这些官方公开记录与对应协议测试通过；不等于获得 CAVP/FIPS 认证。功耗、长期可靠性和侧信道安全需要另外测量。
